# Module 26 — Exercise 1: HTTP Client Basics with httpx

In this exercise, you will practice interacting with REST APIs using `httpx`. You will configure clients with explicit timeouts, supply custom headers, parse structured JSON responses, and handle HTTP status errors gracefully.

| Detail | Value |
|---|---|
| **Time** | 30 minutes |
| **Prerequisites** | Modules 16, 19, 26 README |



## 1. Creating a Persistent Client Session

A `Client` session maintains a connection pool, allowing TCP and TLS reuse across requests.


In [ ]:
import httpx

client = httpx.Client(
    headers={"User-Agent": "PythonTrainingLab/1.0", "Accept": "application/json"},
    timeout=httpx.Timeout(5.0, connect=2.0)
)
print("Client session configured:", client)



## 2. Deliberate Failure: Handling Non-200 Status Codes

Calling `response.raise_for_status()` raises an `httpx.HTTPStatusError` if the response indicates an error (4xx or 5xx).


In [ ]:
# Simulating error handling pattern
try:
    # When status >= 400, raise_for_status raises HTTPStatusError
    response = httpx.Response(status_code=404, request=httpx.Request("GET", "https://api.example.com/missing"))
    response.raise_for_status()
except httpx.HTTPStatusError as exc:
    print(f"Caught expected HTTPStatusError: {exc.response.status_code}")



# Your turn

Complete the tasks below. Do not remove the `# ANSWER n` markers.


### Task 1: Build a safe JSON fetcher

Implement `safe_fetch_json(client, url)` that makes a GET request, raises for status, and returns parsed JSON. If an `httpx.HTTPStatusError` or `httpx.RequestError` occurs, catch it and return `None`.


In [ ]:
# ANSWER 1
def safe_fetch_json(client: httpx.Client, url: str) -> dict | list | None:
    try:
        resp = client.get(url)
        resp.raise_for_status()
        return resp.json()
    except (httpx.HTTPStatusError, httpx.RequestError):
        return None



### Task 2: Paginated Fetcher

Implement `fetch_pages(client, base_url, total_pages)` that requests pages `1` through `total_pages` using query parameter `?page=n` and accumulates all items from each page's `'items'` list into a single list.


In [ ]:
# ANSWER 2
def fetch_pages(client: httpx.Client, base_url: str, total_pages: int) -> list:
    all_items = []
    for page in range(1, total_pages + 1):
        data = safe_fetch_json(client, f"{base_url}?page={page}")
        if data and isinstance(data, dict) and "items" in data:
            all_items.extend(data["items"])
    return all_items



## Self-Check Harness


In [ ]:
def check(passed: bool, msg: str) -> bool:
    status = "PASS" if passed else "FAIL"
    print(f"{status}  {msg}")
    return passed

class MockResponse:
    def __init__(self, status_code, data):
        self.status_code = status_code
        self._data = data
    def raise_for_status(self):
        if self.status_code >= 400:
            raise httpx.HTTPStatusError("Error", request=None, response=self)
    def json(self):
        return self._data

class MockClient:
    def get(self, url):
        if "error" in url:
            return MockResponse(500, {})
        if "page=1" in url:
            return MockResponse(200, {"items": ["item1", "item2"]})
        if "page=2" in url:
            return MockResponse(200, {"items": ["item3", "item4"]})
        return MockResponse(200, {"status": "ok", "items": ["a", "b"]})

mock = MockClient()
results = [
    check(safe_fetch_json(mock, "https://api.example.com/ok") == {"status": "ok", "items": ["a", "b"]}, "Task 1: safe_fetch_json returns valid data"),
    check(safe_fetch_json(mock, "https://api.example.com/error") is None, "Task 1: safe_fetch_json returns None on error"),
    check(fetch_pages(mock, "https://api.example.com/data", 2) == ["item1", "item2", "item3", "item4"], "Task 2: fetch_pages accumulates pagination items"),
]
print(f"Summary: {sum(results)}/{len(results)} checks passed.")

